# 05A-QC — Soil and Lithology Quality Control

**FYP:** Developing a Semi-Dynamic Landslide Susceptibility Decision-Support Dashboard Using Machine Learning for Malaysia

This notebook is a focused QC step after `05A_static_feature_extraction.ipynb`.

It does **not** re-query Google Earth Engine or regenerate background points.

It uses the Stage-05A outputs to answer two questions:

1. Are zero-valued SoilGrids samples real values or background/NoData artefacts?
2. Does the JMG lithology service provide usable, interpretable nationwide coverage?

## Inputs expected

```text
Landslide_FYP_Development/
└── 03_derived_features/
    └── static_features/
        ├── static_feature_samples.geojson
        ├── static_feature_samples.csv
        ├── jmg_lithology_all_layers.geojson
        └── jmg_lithology_field_inventory.csv
```

## Outputs

This QC notebook creates:
- soil zero-pattern audit
- corrected soil-validity flag
- lithology field profiling
- lithology coverage statistics
- lithology coverage map
- revised static-feature validity summary
- cleaned QC version of the static sample table

No original Stage-05A output is overwritten.


In [10]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=UserWarning)

print("pandas:", pd.__version__)
print("geopandas:", gpd.__version__)


pandas: 3.0.2
geopandas: 1.1.4


## 1. Project paths

In [11]:
PROJECT_ROOT = Path(r"C:\Landslide_FYP")

DEV_DIR = PROJECT_ROOT / "Landslide_FYP_Development"

STATIC_DIR = (
    DEV_DIR
    / "03_derived_features"
    / "static_features"
)

SAMPLES_PATH = STATIC_DIR / "static_feature_samples.geojson"
LITHOLOGY_PATH = STATIC_DIR / "jmg_lithology_all_layers.geojson"
LITH_FIELD_PATH = STATIC_DIR / "jmg_lithology_field_inventory.csv"

OUTPUT_DIR = (
    DEV_DIR
    / "03_derived_features"
    / "static_features_qc"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

required = {
    "Static feature samples": SAMPLES_PATH,
    "JMG lithology polygons": LITHOLOGY_PATH,
    "JMG lithology field inventory": LITH_FIELD_PATH,
}

for name, path in required.items():
    if not path.exists():
        raise FileNotFoundError(f"{name} not found:\n{path}")
    print("FOUND:", name, "->", path)

print("\nQC output folder:")
print(OUTPUT_DIR)


FOUND: Static feature samples -> C:\Landslide_FYP\Landslide_FYP_Development\03_derived_features\static_features\static_feature_samples.geojson
FOUND: JMG lithology polygons -> C:\Landslide_FYP\Landslide_FYP_Development\03_derived_features\static_features\jmg_lithology_all_layers.geojson
FOUND: JMG lithology field inventory -> C:\Landslide_FYP\Landslide_FYP_Development\03_derived_features\static_features\jmg_lithology_field_inventory.csv

QC output folder:
C:\Landslide_FYP\Landslide_FYP_Development\03_derived_features\static_features_qc


## 2. Load Stage-05A outputs

In [12]:
samples = gpd.read_file(SAMPLES_PATH).to_crs("EPSG:4326")
lithology = gpd.read_file(LITHOLOGY_PATH).to_crs("EPSG:4326")
field_inventory = pd.read_csv(LITH_FIELD_PATH)

print("Samples:", len(samples))
print("Lithology polygons:", len(lithology))
print("Lithology fields:", len(field_inventory))

display(
    samples[
        [
            "sample_id",
            "sample_class",
            "soil_clay",
            "soil_sand",
            "soil_silt",
            "soil_bulk_density",
            "lithology_raw",
            "lithology_available",
        ]
    ].head()
)


Samples: 3200
Lithology polygons: 18742
Lithology fields: 23


,sample_id,sample_class,soil_clay,soil_sand,soil_silt,soil_bulk_density,lithology_raw,lithology_available
0,POS_0001,landslide,400,330,270,116,9500.0,True
1,POS_0002,landslide,368,351,281,104,9500.0,True
2,POS_0003,landslide,0,0,0,0,NaN,True
3,POS_0006,landslide,345,414,241,111,9500.0,True
4,POS_0007,landslide,0,0,0,0,9500.0,True


# PART A — SOIL ZERO / NODATA QC

## 3. Audit zero values across all four SoilGrids variables

The Stage-05A extraction reported no formal NoData values, but zero values appeared in the sampled outputs.

This step checks whether zeros occur individually or as suspicious all-zero combinations.


In [13]:
soil_cols = [
    "soil_clay",
    "soil_sand",
    "soil_silt",
    "soil_bulk_density",
]

for col in soil_cols:
    samples[col] = pd.to_numeric(samples[col], errors="coerce")
    samples[f"{col}_is_zero"] = samples[col].eq(0)

samples["soil_zero_count"] = samples[
    [f"{c}_is_zero" for c in soil_cols]
].sum(axis=1)

samples["soil_all_zero"] = samples["soil_zero_count"].eq(4)

zero_pattern_summary = (
    samples["soil_zero_count"]
    .value_counts()
    .sort_index()
    .rename_axis("number_of_zero_soil_variables")
    .reset_index(name="sample_count")
)

zero_pattern_summary["sample_pct"] = (
    zero_pattern_summary["sample_count"] / len(samples) * 100
).round(2)

display(zero_pattern_summary)

zero_pattern_summary.to_csv(
    OUTPUT_DIR / "soil_zero_pattern_summary.csv",
    index=False
)


,number_of_zero_soil_variables,sample_count,sample_pct
0,0,3018,94.31
1,1,21,0.66
2,3,13,0.41
3,4,148,4.62


## 4. Inspect zero combinations by class

In [14]:
zero_by_class = (
    samples.groupby(["sample_class", "soil_zero_count"], dropna=False)
    .size()
    .reset_index(name="records")
)

display(zero_by_class)

zero_by_class.to_csv(
    OUTPUT_DIR / "soil_zero_pattern_by_sample_class.csv",
    index=False
)


,sample_class,soil_zero_count,records
0,background_candidate,0,2907
1,background_candidate,1,15
2,background_candidate,3,8
3,background_candidate,4,70
4,landslide,0,111
5,landslide,1,6
6,landslide,3,5
7,landslide,4,78


## 5. Inspect all-zero soil samples

If clay, sand, silt and bulk density are all exactly zero at the same point, that is highly suspicious for a background/NoData artefact.

This notebook treats **all-four-zero** as invalid by default.

Single-variable zero values are not automatically removed.


In [15]:
all_zero_samples = samples.loc[
    samples["soil_all_zero"],
    [
        "sample_id",
        "sample_class",
        "longitude",
        "latitude",
        "soil_clay",
        "soil_sand",
        "soil_silt",
        "soil_bulk_density",
        "geometry",
    ]
].copy()

print("All-four-zero soil samples:", len(all_zero_samples))

display(all_zero_samples.drop(columns="geometry").head(30))

all_zero_samples.drop(columns="geometry").to_csv(
    OUTPUT_DIR / "soil_all_zero_samples.csv",
    index=False
)


All-four-zero soil samples: 148


,sample_id,sample_class,longitude,latitude,soil_clay,soil_sand,soil_silt,soil_bulk_density
2,POS_0003,landslide,116.575400,5.991300,0,0,0,0
4,POS_0007,landslide,101.687000,3.087900,0,0,0,0
5,POS_0008,landslide,101.151900,4.312000,0,0,0,0
6,POS_0009,landslide,100.279172,5.380758,0,0,0,0
7,POS_0010,landslide,101.086000,4.557200,0,0,0,0
12,POS_0017,landslide,100.303200,5.435800,0,0,0,0
13,POS_0018,landslide,101.700300,3.134700,0,0,0,0
15,POS_0020,landslide,102.002500,2.698500,0,0,0,0
16,POS_0021,landslide,101.732900,3.053600,0,0,0,0
17,POS_0022,landslide,101.677787,3.185783,0,0,0,0


## 6. Corrected soil-validity rule

Default QC rule:

```text
soil_valid_qc =
    all four soil values are present
    AND
    NOT all four values are zero
```

This is conservative and avoids pretending all-zero pixels are meaningful soil observations.

If later source metadata proves zero is a valid value for a specific product, this rule can be revised.


In [16]:
samples["soil_complete_qc"] = samples[soil_cols].notna().all(axis=1)

samples["soil_valid_qc"] = (
    samples["soil_complete_qc"]
    & ~samples["soil_all_zero"]
)

soil_qc_summary = pd.DataFrame({
    "check": [
        "Total samples",
        "Soil complete (non-missing)",
        "All-four-zero soil samples",
        "Soil valid after QC",
    ],
    "value": [
        len(samples),
        int(samples["soil_complete_qc"].sum()),
        int(samples["soil_all_zero"].sum()),
        int(samples["soil_valid_qc"].sum()),
    ]
})

display(soil_qc_summary)

soil_qc_summary.to_csv(
    OUTPUT_DIR / "soil_qc_summary.csv",
    index=False
)


,check,value
0,Total samples,3200
1,Soil complete (non-missing),3200
2,All-four-zero soil samples,148
3,Soil valid after QC,3052


## 7. Map suspicious all-zero soil samples

In [17]:
fig, ax = plt.subplots(figsize=(12, 9))

samples.loc[~samples["soil_all_zero"]].plot(
    ax=ax,
    markersize=5,
    alpha=0.25,
    label="No all-zero soil pattern"
)

if samples["soil_all_zero"].any():
    samples.loc[samples["soil_all_zero"]].plot(
        ax=ax,
        marker="x",
        markersize=45,
        label="All four soil values = 0"
    )

ax.set_title("Soil Zero-Value QC")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.legend()
ax.grid(True, alpha=0.2)

plt.tight_layout()

soil_map_path = OUTPUT_DIR / "soil_zero_qc_map.png"
fig.savefig(soil_map_path, dpi=180, bbox_inches="tight")
plt.close(fig)

print("Saved:", soil_map_path)


Saved: C:\Landslide_FYP\Landslide_FYP_Development\03_derived_features\static_features_qc\soil_zero_qc_map.png


# PART B — LITHOLOGY FIELD QC

## 8. Review JMG field inventory

The Stage-05A notebook automatically identified only obvious names such as `AGE` / `AGECODE`.

This QC step profiles **all non-geometry fields** because abbreviated JMG fields may contain the actual geology/lithology information.


In [18]:
display(field_inventory)

print("All lithology polygon columns:")
print(lithology.columns.tolist())


,column,likely_lithology_related
0,OBJECTID,False
1,AGECODE,True
2,AGE,True
3,FCD,False
4,FNM,False
5,UGI,False
6,GLN,False
7,GAM,False
8,GAX,False
9,GLR,False


All lithology polygon columns:
['OBJECTID', 'AGECODE', 'AGE', 'FCD', 'FNM', 'UGI', 'GLN', 'GAM', 'GAX', 'GLR', 'GLS', 'GEO', 'DAT', 'GFD', 'GLL', 'HOR', 'NAM', 'ARK', 'Shape__Area', 'Shape__Length', '_jmg_layer_id', '_jmg_layer_name', 'GlobalID', 'geometry']


## 9. Profile non-null counts and number of unique values

Useful descriptive fields often have:
- substantial non-null coverage
- multiple but not thousands of unique categorical values

This profile helps identify fields such as rock unit, formation, geology, lithology or age without guessing from abbreviations alone.


In [19]:
profile_rows = []

for col in lithology.columns:
    if col == "geometry":
        continue

    series = lithology[col]

    non_null = int(series.notna().sum())
    unique = int(series.dropna().astype(str).nunique())

    examples = (
        series.dropna()
        .astype(str)
        .drop_duplicates()
        .head(8)
        .tolist()
    )

    profile_rows.append({
        "field": col,
        "non_null_count": non_null,
        "non_null_pct": round(non_null / len(lithology) * 100, 2),
        "unique_values": unique,
        "example_values": " | ".join(examples),
    })

lith_field_profile = pd.DataFrame(profile_rows).sort_values(
    ["non_null_pct", "unique_values"],
    ascending=[False, True]
)

display(lith_field_profile)

lith_field_profile.to_csv(
    OUTPUT_DIR / "jmg_lithology_field_profile.csv",
    index=False
)


,field,non_null_count,non_null_pct,unique_values,example_values
3,FCD,18742,100.00,2,GF5000 | GF6000
4,FNM,18742,100.00,2,Geological Lithostratigraphic Unit | Geologica...
16,NAM,18742,100.00,15,Perlis | Kedah | Pulau Pinang | Perak | Kelant...
7,GAM,18742,100.00,23,210 | 420 | 440 | 330 | 220 | 410 | 430 | 460
8,GAX,18742,100.00,28,210 | 420 | 450 | 330 | 220 | 410 | 440 | 460
20,_jmg_layer_id,18742,100.00,30,0 | 1 | 2 | 3 | 4 | 5 | 6 | 7
21,_jmg_layer_name,18742,100.00,30,Lithology of Perlis | Rock age of Perlis | Lit...
5,UGI,18742,100.00,178,130-1000 | 265-6000 | 230-8000 | 230-4000 | 41...
0,OBJECTID,18742,100.00,1599,1 | 2 | 3 | 4 | 5 | 6 | 7 | 8
18,Shape__Area,18742,100.00,9370,127928.415257 | 1442099.655822 | 231274.779188...


## 10. Inspect likely candidate fields in detail

The list below includes both descriptive names and the abbreviated fields observed in the JMG service.

No field is automatically declared the final lithology predictor here.


In [20]:
candidate_fields = [
    "AGE",
    "AGECODE",
    "GLN",
    "GAM",
    "GAX",
    "GLR",
    "GLS",
    "GEO",
    "GLL",
    "NAM",
]

candidate_fields = [
    col for col in candidate_fields
    if col in lithology.columns
]

print("Candidate fields present:")
print(candidate_fields)

candidate_value_rows = []

for col in candidate_fields:
    counts = (
        lithology[col]
        .astype("string")
        .fillna("<MISSING>")
        .value_counts(dropna=False)
        .head(30)
    )

    for value, count in counts.items():
        candidate_value_rows.append({
            "field": col,
            "value": value,
            "count": int(count),
        })

candidate_values = pd.DataFrame(candidate_value_rows)

display(candidate_values)

candidate_values.to_csv(
    OUTPUT_DIR / "jmg_candidate_field_value_counts.csv",
    index=False
)


Candidate fields present:
['AGE', 'AGECODE', 'GLN', 'GAM', 'GAX', 'GLR', 'GLS', 'GEO', 'GLL', 'NAM']


,field,value,count
0,AGE,<MISSING>,5656
1,AGE,Quaternary,2794
2,AGE,Triassic,2666
3,AGE,Igneous Activities: Triassic,2286
4,AGE,Carboniferous,1254
...,...,...,...
181,NAM,Melaka,280
182,NAM,Pulau Pinang,164
183,NAM,Perlis,156
184,NAM,WP Kuala Lumpur,28


# PART C — LITHOLOGY SPATIAL COVERAGE

## 11. Measure actual polygon coverage of the 3,200 samples

This distinguishes:
- missing descriptive attribute
from
- no JMG polygon covering the sample location

That distinction is critical.


In [21]:
coverage_join = gpd.sjoin(
    samples[
        [
            "sample_id",
            "sample_class",
            "geometry"
        ]
    ],
    lithology[
        [
            "_jmg_layer_id",
            "_jmg_layer_name",
            "geometry"
        ]
    ],
    how="left",
    predicate="within"
)

coverage_counts = (
    coverage_join
    .groupby("sample_id")["_jmg_layer_id"]
    .apply(lambda s: int(s.notna().sum()))
)

samples["lithology_polygon_coverage_count"] = (
    samples["sample_id"]
    .map(coverage_counts)
    .fillna(0)
    .astype(int)
)

samples["lithology_spatially_covered"] = (
    samples["lithology_polygon_coverage_count"] > 0
)

coverage_summary = pd.DataFrame({
    "check": [
        "Total samples",
        "Spatially inside >=1 JMG lithology polygon",
        "No JMG lithology polygon coverage",
        "Covered by >1 polygon",
    ],
    "value": [
        len(samples),
        int(samples["lithology_spatially_covered"].sum()),
        int((~samples["lithology_spatially_covered"]).sum()),
        int((samples["lithology_polygon_coverage_count"] > 1).sum()),
    ]
})

display(coverage_summary)

coverage_summary.to_csv(
    OUTPUT_DIR / "jmg_lithology_spatial_coverage_summary.csv",
    index=False
)


,check,value
0,Total samples,3200
1,Spatially inside >=1 JMG lithology polygon,3166
2,No JMG lithology polygon coverage,34
3,Covered by >1 polygon,3166


## 12. Coverage by sample class

In [22]:
coverage_by_class = (
    samples.groupby(
        ["sample_class", "lithology_spatially_covered"],
        dropna=False
    )
    .size()
    .reset_index(name="records")
)

display(coverage_by_class)

coverage_by_class.to_csv(
    OUTPUT_DIR / "jmg_lithology_coverage_by_sample_class.csv",
    index=False
)


,sample_class,lithology_spatially_covered,records
0,background_candidate,False,32
1,background_candidate,True,2968
2,landslide,False,2
3,landslide,True,198


## 13. Coverage by JMG source layer

In [23]:
layer_match = coverage_join.loc[
    coverage_join["_jmg_layer_id"].notna(),
    [
        "sample_id",
        "_jmg_layer_id",
        "_jmg_layer_name"
    ]
].drop_duplicates()

layer_sample_counts = (
    layer_match.groupby(
        ["_jmg_layer_id", "_jmg_layer_name"],
        dropna=False
    )["sample_id"]
    .nunique()
    .reset_index(name="covered_samples")
    .sort_values("covered_samples", ascending=False)
)

display(layer_sample_counts)

layer_sample_counts.to_csv(
    OUTPUT_DIR / "jmg_lithology_sample_coverage_by_layer.csv",
    index=False
)


,_jmg_layer_id,_jmg_layer_name,covered_samples
26,28.0,Lithology of Sarawak,1118
27,29.0,Rock age of Sarawak,1118
24,26.0,Lithology of Sabah,698
25,27.0,Rock age of Sabah,698
13,13.0,Rock age of Pahang,364
12,12.0,Lithology of Pahang,364
6,6.0,Lithology of Perak,190
7,7.0,Rock age of Perak,190
22,24.0,Lithology of Johor,173
23,25.0,Rock age of Johor,173


## 14. Lithology spatial coverage map

In [24]:
fig, ax = plt.subplots(figsize=(12, 9))

samples.loc[samples["lithology_spatially_covered"]].plot(
    ax=ax,
    markersize=6,
    alpha=0.35,
    label="Inside JMG lithology polygon"
)

samples.loc[~samples["lithology_spatially_covered"]].plot(
    ax=ax,
    marker="x",
    markersize=28,
    alpha=0.7,
    label="No JMG lithology polygon"
)

ax.set_title("JMG Lithology Spatial Coverage QC")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.legend()
ax.grid(True, alpha=0.2)

plt.tight_layout()

lith_map_path = OUTPUT_DIR / "jmg_lithology_coverage_map.png"
fig.savefig(lith_map_path, dpi=180, bbox_inches="tight")
plt.close(fig)

print("Saved:", lith_map_path)


Saved: C:\Landslide_FYP\Landslide_FYP_Development\03_derived_features\static_features_qc\jmg_lithology_coverage_map.png


# PART D — REVISED STATIC VALIDITY

## 15. Revised validity logic

For the Malaysia-wide baseline:

```text
national_core_static_valid =
    terrain valid
    AND
    soil valid after QC
    AND
    fault distance available
```

Lithology is **not mandatory** unless the QC proves near-national coverage.

This prevents dropping large parts of Malaysia simply because the API polygon coverage is incomplete.


In [25]:
samples["national_core_static_valid"] = (
    samples["terrain_valid"]
    & samples["soil_valid_qc"]
    & samples["fault_distance_available"]
)

samples["geology_enhanced_valid"] = (
    samples["national_core_static_valid"]
    & samples["lithology_spatially_covered"]
)

revised_validity_summary = pd.DataFrame({
    "check": [
        "Total samples",
        "Terrain valid",
        "Soil valid after QC",
        "Fault distance available",
        "National core-static valid",
        "Lithology spatially covered",
        "Geology-enhanced valid",
    ],
    "value": [
        len(samples),
        int(samples["terrain_valid"].sum()),
        int(samples["soil_valid_qc"].sum()),
        int(samples["fault_distance_available"].sum()),
        int(samples["national_core_static_valid"].sum()),
        int(samples["lithology_spatially_covered"].sum()),
        int(samples["geology_enhanced_valid"].sum()),
    ]
})

display(revised_validity_summary)

revised_validity_summary.to_csv(
    OUTPUT_DIR / "revised_static_validity_summary.csv",
    index=False
)


,check,value
0,Total samples,3200
1,Terrain valid,3200
2,Soil valid after QC,3052
3,Fault distance available,3200
4,National core-static valid,3052
5,Lithology spatially covered,3166
6,Geology-enhanced valid,3028


## 16. Save QC-corrected sample table

This file becomes the preferred Stage-05A static sample table for subsequent work.


In [26]:
qc_geojson = OUTPUT_DIR / "static_feature_samples_qc.geojson"
qc_csv = OUTPUT_DIR / "static_feature_samples_qc.csv"

samples.to_file(
    qc_geojson,
    driver="GeoJSON"
)

samples.drop(
    columns="geometry",
    errors="ignore"
).to_csv(
    qc_csv,
    index=False
)

print("Saved:", qc_geojson)
print("Saved:", qc_csv)


Saved: C:\Landslide_FYP\Landslide_FYP_Development\03_derived_features\static_features_qc\static_feature_samples_qc.geojson
Saved: C:\Landslide_FYP\Landslide_FYP_Development\03_derived_features\static_features_qc\static_feature_samples_qc.csv


## 17. Save national-baseline-valid background candidates

These are still **background candidates**, not final pseudo-absences.


In [27]:
national_valid_background = samples.loc[
    (
        samples["sample_class"] == "background_candidate"
    )
    & samples["national_core_static_valid"]
].copy()

national_bg_geojson = (
    OUTPUT_DIR
    / "background_candidates_national_core_valid.geojson"
)

national_bg_csv = (
    OUTPUT_DIR
    / "background_candidates_national_core_valid.csv"
)

national_valid_background.to_file(
    national_bg_geojson,
    driver="GeoJSON"
)

national_valid_background.drop(
    columns="geometry",
    errors="ignore"
).to_csv(
    national_bg_csv,
    index=False
)

print(
    "National-core-valid background candidates:",
    len(national_valid_background)
)


National-core-valid background candidates: 2930


# Interpretation guide

After running this notebook:

### Soil
If most zero-valued samples are all-four-zero together, treat them as invalid background/NoData pixels.

If zeros occur frequently in only one soil variable while other soil variables are realistic, do not automatically remove them without checking source metadata.

### Lithology
If JMG polygon coverage is close to national:
- lithology may be included in the national baseline after identifying the correct attribute field.

If coverage is partial:
- keep lithology out of the Malaysia-wide core baseline
- use it in a geology-enhanced subset/secondary experiment.

### Do not move to model training yet
The next step remains:

`05B_temporal_feature_extraction.ipynb`

for:
- historical land cover
- roads
- waterways
- rainfall


# Outputs to send for review

Please send:

```text
soil_zero_pattern_summary.csv
soil_qc_summary.csv
soil_zero_qc_map.png

jmg_lithology_field_profile.csv
jmg_candidate_field_value_counts.csv
jmg_lithology_spatial_coverage_summary.csv
jmg_lithology_coverage_map.png

revised_static_validity_summary.csv
```

These are enough to decide whether Stage 05A is fully closed.
